# Galatiq E2E Walkthrough

Offline end-to-end invoice pipeline: **Ingestion → Validation → Approval → Payment**.

For the full Case Simulation Contract (W1–W10 + fault injection), open
`e2e_case_simulation.ipynb` or run:

```bash
python main.py --sim=case
```

Optional Grok: set `XAI_API_KEY=xai-your-key-here` in `.env` (see `.env.example`).

In [ ]:
from pathlib import Path
import os
import sys
import tempfile
from dotenv import load_dotenv

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from invoice_automation.orchestration.pipeline import Pipeline

DATA = ROOT / "data" / "invoices"
from invoice_automation.case_brief import is_live_xai_key

XAI_API_KEY = os.getenv("XAI_API_KEY", "your_key")
USE_GROK = is_live_xai_key(XAI_API_KEY)
work = Path(tempfile.mkdtemp(prefix="galatiq_nb_"))
pipe = Pipeline(work_dir=work, use_grok=USE_GROK)
print("work=", work, "use_grok=", USE_GROK)
work

## Happy path (INV-1001)

In [ ]:
ok = pipe.process_file(DATA / "invoice_1001.txt")
ok.status, ok.issues, ok.approval_reason

## Failure modes: overstock, fraud item, unknown SKU

In [ ]:
cases = ["invoice_1002.txt", "invoice_1003.txt", "invoice_1008.txt", "invoice_1009.json"]
for name in cases:
    p = Pipeline(work_dir=work / name, use_grok=False)
    r = p.process_file(DATA / name)
    print(name, "->", r.status, r.issues or r.approval_reason)

## Corpus summary

In [ ]:
summary = Pipeline(work_dir=work / "all", use_grok=False).process_directory(DATA)
for r in summary:
    print(f"{Path(r.source_path).name:28} {r.status:10} {r.invoice_id}")
print("metrics", summary[-1].metrics if summary else {})

## Full Case Simulation Contract

Prefer `e2e_case_simulation.ipynb` for charts + CLI parity. Quick check:

In [ ]:
from invoice_automation.sim.runner import run_case_simulation

report = run_case_simulation(work / "case_sim", data_dir=DATA, use_grok=False)
print(f"{report.passed}/{report.total} ok={report.ok}")
assert report.ok